# w15_gpu2_short_disp_hardq_anchor
Inputs: `shortpose-wave1-private-inputs`


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
import torch

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir() and torch.cuda.is_available(), 'Kaggle GPU required'
REV = 'ed23b99140a5f1fc23b5fd15b7565091b6270257'
with tempfile.TemporaryDirectory(prefix='w15_sd_') as ts:
    tmp = Path(ts)
    repo, meta_dir = tmp / 'repo', tmp / 'pinns_meta'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1'}
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env, check=True)
    dest = WORK / 'pinns_short_disp_hardq_anchor_5seed.zip'
    subprocess.run([sys.executable, '-u', '-m', 'scripts.run_pinns_short_disp_hardq_anchor_5seed', str(meta_dir), str(pose_npz), str(dest)], cwd=repo, env=env, check=True)
    print('DONE:', dest, dest.stat().st_size, hashlib.sha256(dest.read_bytes()).hexdigest(), flush=True)
